# Analyse d'un journal de serveur web avec les RDD

Ce TP travaille avec l'API de plus bas niveau de Spark, les RDD, sur des données qu'aucun schéma ne décrit : le journal d'accès d'un serveur web, une ligne de texte par requête. Vous y créez une session et un RDD, analysez chaque ligne, comptez avec map/reduce, réutilisez un RDD mis en cache, observez les partitions, partagez une variable broadcast et un accumulateur entre les tâches, passez au `DataFrame` et à SQL, puis soumettez un traitement en script avec `spark-submit`.

Chaque exercice s'appuie sur le précédent : faites-les dans l'ordre. Spark tourne ici en mode local, sur les cœurs de la machine de Colab ; le même code tournerait sur un cluster, seuls l'adresse du gestionnaire (`master`) et le chemin des données (sur HDFS ou S3, que tous les exécuteurs lisent) changeraient.

## Installation et données

Le journal est celui d'une boutique en ligne iranienne, [zanbil.ir](https://doi.org/10.7910/DVN/3QBYB5), en janvier 2019 (Farzin Zaker, Harvard Dataverse, licence CC0), réduit au centième : 103 652 lignes, 35 Mo. Chaque ligne suit le format *combined* de nginx et d'Apache : l'adresse IP du client, deux champs vides (`-`), la date entre crochets, la requête entre guillemets (méthode, chemin, protocole), le code de statut HTTP, la taille de la réponse en octets, la page d'où vient le client et son navigateur (*user agent*), entre guillemets.

In [ ]:
!pip install pyspark
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/nginx-log/access-small.log

In [ ]:
import re
import time

from pyspark.sql import SparkSession

## 1. Session et contexte

Créez une session Spark nommée `journal-web`, qui tourne en local sur tous les cœurs de la machine, avec `SparkSession.builder`, `appName`, `master` et `getOrCreate`. Récupérez son contexte, `sc`, celui des RDD. Affichez l'adresse du gestionnaire (`master`), le parallélisme par défaut (`defaultParallelism`), la version de Spark et l'adresse de son interface web (`uiWebUrl`).

Combien de tâches Spark peut-il exécuter à la fois ici ?

Demandez ensuite une autre session, `SparkSession.builder.appName("autre").getOrCreate()`, dans la variable `autre`. Vérifiez que c'est la même session (`autre is spark`), et que le nom de l'application, `sc.appName`, n'a pas changé. Lisez enfin, dans la configuration de la session (`spark.conf.get`), le nombre de partitions qu'aura un `DataFrame` après un shuffle, `spark.sql.shuffle.partitions` : vaut-il le parallélisme par défaut ?


**Documentation utile**

- [`pyspark.sql.SparkSession.conf`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.conf.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`master("local[*]")` crée un exécuteur par cœur ; `local[4]` en fixerait quatre. Le contexte est l'attribut `sparkContext` de la session. Sur Colab, `defaultParallelism` vaut le nombre de cœurs de la machine, 2 en général. `getOrCreate()` renvoie la session déjà ouverte : vous devez lire `True`, puis `journal-web`. `spark.conf.get` prend le nom du réglage et renvoie sa valeur, une chaîne.

```python
spark = SparkSession.builder.appName(...).master(...).getOrCreate()
sc = spark.sparkContext
print(sc.master, sc.defaultParallelism, spark.version, sc.uiWebUrl)

autre = SparkSession.builder.appName(...).getOrCreate()
print(autre is spark, sc.appName)
print(spark.conf.get(...))
```

### Solution

In [ ]:
spark = SparkSession.builder.appName("journal-web").master("local[*]").getOrCreate()
sc = spark.sparkContext
print(sc.master, sc.defaultParallelism, spark.version, sc.uiWebUrl)
# Spark exécute à la fois autant de tâches que defaultParallelism : une par cœur.

autre = SparkSession.builder.appName("autre").getOrCreate()
print(autre is spark, sc.appName)
print(spark.conf.get("spark.sql.shuffle.partitions"))
# getOrCreate() renvoie la session ouverte, une seule par application : son
# contexte garde son nom et son master. Un DataFrame a 200 partitions après un
# shuffle, quel que soit le nombre de cœurs (l'exécution adaptative les regroupe).

## 2. Charger le journal

Créez le RDD `lignes`, une ligne du fichier `access-small.log` par élément, avec `sc.textFile`. Rien n'est encore lu : comme une transformation, `textFile` ne fait que décrire le RDD. Affichez ensuite le nombre de lignes (`count`, vous devez trouver 103 652), le nombre de partitions du RDD (`getNumPartitions`) et les deux premières lignes (`take`).

Pourquoi le fichier n'est-il découpé qu'en peu de partitions ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`take(2)` renvoie une liste de deux chaînes. `textFile` découpe un fichier en blocs (32 Mo pour un fichier local, 128 Mo sur HDFS), avec au moins deux partitions par défaut (`minPartitions`).

```python
lignes = sc.textFile(...)
print(lignes.count(), lignes.getNumPartitions())
lignes.take(...)
```

### Solution

In [ ]:
lignes = sc.textFile("access-small.log")
print(lignes.count(), lignes.getNumPartitions())
lignes.take(2)
# Deux partitions : minPartitions vaut 2 par défaut, et un fichier de 35 Mo, à peine
# plus grand qu'un bloc local de 32 Mo, n'est pas coupé par bloc. Deux
# partitions, c'est deux tâches au plus à la fois, quel que soit le nombre de cœurs.

## 3. Analyser chaque ligne

Une ligne de journal n'est qu'une chaîne : il faut en extraire les champs. L'expression régulière `MOTIF` ci-dessous en capture sept, dans l'ordre : l'adresse IP, la date, la méthode, le chemin, le code de statut, la taille et le navigateur.

Écrivez la fonction `analyser(ligne)`, qui renvoie le tuple `(ip, date, methode, chemin, statut, taille, navigateur)`, avec `statut` et `taille` convertis en entiers (une taille `-` vaut 0), ou `None` quand la ligne ne correspond pas au motif. Créez ensuite le RDD `journal` des lignes analysées, sans les `None`, avec `map` et `filter`, et comptez ses éléments : combien de lignes sont mal formées ?


**Documentation utile**

- [`re.Pattern.match`](https://docs.python.org/3/library/re.html#re.Pattern.match)
- [`re.Match.groups`](https://docs.python.org/3/library/re.html#re.Match.groups)

In [ ]:
MOTIF = re.compile(
  r'(\S+) \S+ \S+ \[([^\]]+)\] "(\S+) (\S+) [^"]*" (\d{3}) (\d+|-) "[^"]*" "([^"]*)"'
)


def analyser(ligne):
  # Votre code ici
  ...

### Conseils supplémentaires

`MOTIF.match(ligne)` renvoie `None` quand la ligne ne correspond pas, sinon un objet dont `groups()` donne les sept champs, des chaînes. Sur la première ligne du journal, `analyser` renvoie `('54.36.149.41', '22/Jan/2019:03:56:14 +0330', 'GET', '/filter/27|13%20…', 200, 30577, 'Mozilla/5.0 (compatible; AhrefsBot/6.1; +http://ahrefs.com/robot/)')`. Vous devez trouver 103 650 lignes analysées.

```python
def analyser(ligne):
  correspondance = MOTIF.match(ligne)
  if correspondance is None:
    return None
  ip, date, methode, chemin, statut, taille, navigateur = correspondance.groups()
  return (ip, date, methode, chemin, ..., ..., navigateur)


journal = lignes.map(...).filter(lambda requete: ...)
print(journal.count())
```

### Solution

In [ ]:
def analyser(ligne):
  correspondance = MOTIF.match(ligne)
  if correspondance is None:
    return None
  ip, date, methode, chemin, statut, taille, navigateur = correspondance.groups()
  taille = 0 if taille == "-" else int(taille)
  return (ip, date, methode, chemin, int(statut), taille, navigateur)


journal = lignes.map(analyser).filter(lambda requete: requete is not None)
print(journal.count())
print(lignes.count() - journal.count(), "lignes mal formées")

## 4. Codes de statut : map/reduce

Comptez les requêtes par code de statut, à la manière d'un comptage de mots : `map` émet une paire `(statut, 1)` par requête, `reduceByKey` additionne les valeurs de chaque clé. Triez le résultat du plus fréquent au moins fréquent avec `sortBy`, puis rapatriez-le avec `collect`.

Quelle part des requêtes finit en erreur (un statut de 400 ou plus) ?


**Documentation utile**

- [`pyspark.RDD.sortBy`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.sortBy.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le statut est le cinquième champ du tuple, `requete[4]`. `sortBy(f, ascending=False)` trie selon `f` appliquée à chaque paire, ici sa valeur, `paire[1]`. Vous devez trouver 95 912 requêtes de statut 200 en tête, puis 3 379 de statut 304 (« non modifié » : le navigateur avait déjà la page).

```python
statuts = journal.map(lambda requete: (..., 1)).reduceByKey(lambda a, b: ...)
statuts.sortBy(lambda paire: ..., ascending=False).collect()
```

### Solution

In [ ]:
statuts = journal.map(lambda requete: (requete[4], 1)).reduceByKey(lambda a, b: a + b)
par_frequence = statuts.sortBy(lambda paire: paire[1], ascending=False).collect()
print(par_frequence)
nb_erreurs = sum(nombre for statut, nombre in par_frequence if statut >= 400)
print(f"{nb_erreurs} erreurs, {nb_erreurs / journal.count():.1%} des requêtes")

## 5. Les rubriques les plus demandées

La rubrique d'une requête est le premier segment de son chemin, sans les paramètres : `/image` pour `/image/4574/productModel/200x200`, `/m` pour `/m/browse/meat-grinder`. Écrivez la fonction `rubrique(chemin)`, puis affichez les dix rubriques les plus demandées, avec leur nombre de requêtes, avec `reduceByKey` et `takeOrdered`.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`chemin.split("/")[1]` donne le premier segment, et `.split("?")[0]` retire ses paramètres. `takeOrdered(10, key=...)` renvoie les dix plus petits éléments selon `key` : une clé négative, `-paire[1]`, donne les dix plus grands. Vous devez trouver `/image` en tête, avec 56 573 requêtes : les pages d'un site demandent bien plus d'images que de pages.

```python
def rubrique(chemin):
  return "/" + ...


rubriques = journal.map(lambda requete: (rubrique(requete[3]), 1)).reduceByKey(...)
rubriques.takeOrdered(10, key=lambda paire: ...)
```

### Solution

In [ ]:
def rubrique(chemin):
  return "/" + chemin.split("/")[1].split("?")[0]


rubriques = journal.map(lambda requete: (rubrique(requete[3]), 1)).reduceByKey(
  lambda a, b: a + b
)
rubriques.takeOrdered(10, key=lambda paire: -paire[1])

## 6. Taille moyenne des réponses par heure

Calculez la taille moyenne des réponses, en octets, pour chaque heure de la journée. L'heure est dans la date, `22/Jan/2019:03:56:14 +0330`, aux caractères 12 et 13. Une moyenne ne se combine pas deux à deux : `reduceByKey` additionne des paires `(somme, nombre)`, puis `mapValues` divise. Triez par heure avec `sortByKey`.

À quelle heure les réponses sont-elles les plus lourdes ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

La clé est `requete[1][12:14]`, la valeur `(requete[5], 1)`. La fonction de `reduceByKey` reçoit deux paires et renvoie leur somme terme à terme. Vous devez trouver 12 381 octets en moyenne à minuit (`'00'`).

```python
tailles = journal.map(lambda requete: (requete[1][12:14], (requete[5], 1)))
sommes = tailles.reduceByKey(lambda a, b: (a[0] + b[0], ...))
moyennes = sommes.mapValues(lambda somme_nombre: ...).sortByKey()
moyennes.collect()
```

### Solution

In [ ]:
tailles = journal.map(lambda requete: (requete[1][12:14], (requete[5], 1)))
sommes = tailles.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
moyennes = sommes.mapValues(lambda somme_nombre: somme_nombre[0] / somme_nombre[1])
for heure, moyenne in moyennes.sortByKey().collect():
  print(heure, round(moyenne))
# Les réponses les plus lourdes, de 19 000 à 21 800 octets, tombent entre 3 h et
# 6 h, quand le trafic est le plus faible (moins de 1 000 requêtes par heure,
# contre plus de 7 000 vers midi). L'exercice 9 dira qui consulte le site à ces
# heures-là.

## 7. Réutiliser un RDD : le cache

Chaque action recalcule `journal` depuis le fichier : lecture, `map(analyser)` et `filter`. Mesurez la durée de deux `count()` successifs sur `journal` avec `time.perf_counter`, puis mettez `journal` en cache avec `cache()` et mesurez encore deux `count()`.

Pourquoi le premier `count()` après `cache()` n'est-il pas plus rapide ? Que montre ensuite `journal.toDebugString()` ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`cache()` est paresseux, comme une transformation : c'est la première action qui calcule le RDD et le garde en mémoire. `toDebugString()` renvoie des octets : `.decode()` les rend lisibles, et sa ligne `CachedPartitions` donne la mémoire occupée.

```python
def chronometrer(rdd):
  debut = time.perf_counter()
  rdd.count()
  return time.perf_counter() - debut


print(chronometrer(journal), chronometrer(journal))
journal.cache()
print(...)
print(journal.toDebugString().decode())
```

### Solution

In [ ]:
def chronometrer(rdd):
  debut = time.perf_counter()
  rdd.count()
  return time.perf_counter() - debut


print("Sans cache :", chronometrer(journal), chronometrer(journal))
journal.cache()
print("Avec cache :", chronometrer(journal), chronometrer(journal))
print(journal.toDebugString().decode())
# Le premier count() après cache() lit le fichier et analyse chaque ligne, comme
# avant, en gardant en plus le résultat en mémoire ; le second ne relit que la
# mémoire. Le lignage montre les partitions en cache (CachedPartitions) au-dessus
# des étapes qui les recalculeraient.

## 8. Partitions

Affichez le nombre d'éléments de chaque partition de `journal`, avec `glom` (une liste par partition) puis `map(len)`. Faites de même après `repartition(8)` et après `coalesce(1)`. Comptez enfin les requêtes de chaque partition de `journal` sans `glom`, avec `mapPartitions`, qui reçoit toute une partition à la fois.

Affichez pour finir le lignage (`toDebugString`) de `journal.repartition(8)` et celui de `journal.coalesce(1)` : lequel commence une nouvelle étape, et donc échange des données entre les exécuteurs (un shuffle) ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

La fonction de `mapPartitions` reçoit un itérateur sur les éléments de la partition et doit renvoyer un itérable : `[sum(1 for _ in partition)]`, une liste d'un seul nombre. Vous devez trouver deux partitions d'environ 51 800 requêtes pour `journal`, et huit d'environ 12 950 après `repartition(8)`. Dans le lignage, une nouvelle étape se voit à un retrait `+-`, devant le nombre de partitions de l'étape d'avant, et le RDD de l'échange s'appelle `ShuffledRDD`.

```python
print(journal.glom().map(len).collect())
print(journal.repartition(...).glom().map(len).collect())
print(journal.coalesce(...).glom().map(len).collect())
print(journal.mapPartitions(lambda partition: [...]).collect())
print(journal.repartition(8).toDebugString().decode())
print(...)
```

### Solution

In [ ]:
print(journal.glom().map(len).collect())
print(journal.repartition(8).glom().map(len).collect())
print(journal.coalesce(1).glom().map(len).collect())
print(journal.mapPartitions(lambda partition: [sum(1 for _ in partition)]).collect())
print(journal.repartition(8).toDebugString().decode())
print(journal.coalesce(1).toDebugString().decode())
# Le lignage de repartition a un ShuffledRDD et un retrait +-(2) : une nouvelle
# étape, qui redistribue tous les éléments par un shuffle, pour des partitions
# équilibrées. Celui de coalesce n'a qu'un CoalescedRDD, posé sur les partitions
# en cache : il fusionne des partitions voisines sans shuffle, et ne sert qu'à en
# réduire le nombre.

## 9. Variable broadcast : les robots

Les robots d'indexation se reconnaissent à leur navigateur, qui contient `bot`, `crawler` ou `spider` (`AhrefsBot`, `Googlebot`). Envoyez cette liste de mots à tous les exécuteurs une seule fois, comme variable broadcast (`sc.broadcast`), écrivez la fonction `est_robot(navigateur)`, qui lit la liste par `.value`, puis comptez les requêtes des robots et des humains avec `reduceByKey`.

Quelle part du trafic vient des robots ?

Comptez enfin les requêtes par clé `(heure, est_robot(...))`, l'heure prise comme à l'exercice 6, et affichez pour chaque heure la part des robots. Le journal est à l'heure de Téhéran (`+0330`) : qui consulte le site entre 3 h et 6 h, quand les réponses sont les plus lourdes ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le navigateur est le septième champ, `requete[6]`, à mettre en minuscules (`lower()`) avant de chercher les mots. `any(mot in texte for mot in mots)` vaut `True` si l'un des mots y est. Vous devez trouver 11 180 requêtes de robots. Pour la part par heure, `collect` rapatrie les 48 paires `((heure, robot), nombre)` dans un dictionnaire : vous devez trouver 36 % de robots à 3 h et 48 % à 5 h.

```python
mots_robots = sc.broadcast([...])


def est_robot(navigateur):
  navigateur = navigateur.lower()
  return any(mot in navigateur for mot in ...)


journal.map(lambda requete: (est_robot(requete[6]), 1)).reduceByKey(...).collect()

par_heure = journal.map(lambda requete: ((..., ...), 1)).reduceByKey(...)
nombres_par_heure = dict(par_heure.collect())
for heure in sorted({heure for heure, robot in nombres_par_heure}):
  nb_robots = nombres_par_heure.get((heure, True), 0)
  ...
```

### Solution

In [ ]:
mots_robots = sc.broadcast(["bot", "crawler", "spider"])


def est_robot(navigateur):
  navigateur = navigateur.lower()
  return any(mot in navigateur for mot in mots_robots.value)


robots = journal.map(lambda requete: (est_robot(requete[6]), 1)).reduceByKey(
  lambda a, b: a + b
)
nombres = dict(robots.collect())
print(nombres, f"{nombres[True] / journal.count():.1%} de robots")

par_heure = journal.map(
  lambda requete: ((requete[1][12:14], est_robot(requete[6])), 1)
).reduceByKey(lambda a, b: a + b)
nombres_par_heure = dict(par_heure.collect())
for heure in sorted({heure for heure, robot in nombres_par_heure}):
  nb_robots = nombres_par_heure.get((heure, True), 0)
  nb_humains = nombres_par_heure.get((heure, False), 0)
  print(heure, f"{nb_robots / (nb_robots + nb_humains):.0%} de robots")
# De 3 h à 6 h, quand le trafic est le plus faible, les robots d'indexation font
# 36 à 48 % des requêtes, contre 7 à 12 % de 9 h à minuit. Or les robots demandent
# surtout des pages (26 Ko en moyenne), et les humains surtout les images et les
# fichiers statiques qu'elles chargent (11 Ko) : la nuit, la réponse moyenne s'alourdit.

## 10. Accumulateur : les lignes mal formées

Comptez les lignes mal formées pendant l'analyse, sans `filter` ni `count` à part. Essayez d'abord avec une variable du notebook : `compteur = 0`, une fonction `compter(ligne)` qui ajoute 1 à `compteur` (déclaré `global`) quand `analyser` renvoie `None`, puis `lignes.foreach(compter)`. Que vaut `compteur` ensuite, et pourquoi ?

Créez un accumulateur `nb_mal_formees` (`sc.accumulator`), et une fonction `analyser_et_compter(ligne)` qui appelle `analyser` et ajoute 1 à l'accumulateur quand elle renvoie `None`. Créez le RDD `lignes.map(analyser_et_compter)`, appelez deux fois `count()` sur lui, et affichez la valeur de l'accumulateur après chacun.

Pourquoi le total double-t-il ? Refaites le compte dans une action, `foreach`, avec un accumulateur neuf : quel total trouvez-vous ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`global compteur`, en tête de `compter`, lui fait modifier la variable du module au lieu d'en créer une locale ; vous devez lire 0. `nb_mal_formees.add(1)` incrémente l'accumulateur dans une tâche, `nb_mal_formees.value` le lit, dans le pilote seulement. Vous devez lire 2, puis 4 ; avec `foreach`, 2.

```python
compteur = 0


def compter(ligne):
  global compteur
  if ...:
    compteur += 1


lignes.foreach(compter)
print(compteur)

nb_mal_formees = sc.accumulator(0)


def analyser_et_compter(ligne):
  requete = analyser(ligne)
  if requete is None:
    ...
  return requete


analysees = lignes.map(analyser_et_compter)
analysees.count()
print(nb_mal_formees.value)
...

nb_dans_action = sc.accumulator(0)
lignes.foreach(lambda ligne: nb_dans_action.add(1) if ... else None)
print(nb_dans_action.value)
```

### Solution

In [ ]:
compteur = 0


def compter(ligne):
  global compteur
  if analyser(ligne) is None:
    compteur += 1


lignes.foreach(compter)
print(compteur)
# 0 : compter part chez les exécuteurs avec une copie de compteur, que chaque
# tâche incrémente ; le compteur du pilote ne change pas.

nb_mal_formees = sc.accumulator(0)


def analyser_et_compter(ligne):
  requete = analyser(ligne)
  if requete is None:
    nb_mal_formees.add(1)
  return requete


analysees = lignes.map(analyser_et_compter)
analysees.count()
print(nb_mal_formees.value)
analysees.count()
print(nb_mal_formees.value)
# analysees n'est pas en cache : le second count() refait le map, et chaque ligne
# mal formée est comptée une seconde fois.

nb_dans_action = sc.accumulator(0)
lignes.foreach(lambda ligne: nb_dans_action.add(1) if analyser(ligne) is None else None)
print(nb_dans_action.value)

## 11. Du RDD au `DataFrame`

Les données sont maintenant structurées : passez à Spark SQL. Créez le `DataFrame` `requetes` depuis `journal` avec `toDF` et les noms de colonnes `ip`, `date`, `methode`, `chemin`, `statut`, `taille` et `navigateur`, affichez son schéma (`printSchema`), puis enregistrez-le comme vue (`createOrReplaceTempView`) et calculez en SQL (`spark.sql`) le nombre de requêtes par statut, du plus fréquent au moins fréquent. Comparez avec l'exercice 4.

Revenez enfin au RDD avec l'attribut `rdd` du `DataFrame` : quel est le type de ses éléments ?


**Documentation utile**

- [`pyspark.sql.DataFrame.createOrReplaceTempView`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.createOrReplaceTempView.html)
- [`pyspark.sql.SparkSession.sql`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.sql.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`journal.toDF([...])` déduit les types des tuples : `statut` et `taille` deviennent des `long`. La requête : `SELECT statut, COUNT(*) AS nombre FROM journal GROUP BY statut ORDER BY nombre DESC`, et `.show()` affiche son résultat.

```python
requetes = journal.toDF([...])
requetes.printSchema()
requetes.createOrReplaceTempView("journal")
spark.sql("SELECT ... FROM journal GROUP BY ... ORDER BY ...").show()
requetes.rdd.first()
```

### Solution

In [ ]:
requetes = journal.toDF(
  ["ip", "date", "methode", "chemin", "statut", "taille", "navigateur"]
)
requetes.printSchema()
requetes.createOrReplaceTempView("journal")
spark.sql(
  "SELECT statut, COUNT(*) AS nombre FROM journal GROUP BY statut ORDER BY nombre DESC"
).show()
requetes.rdd.first()
# Les mêmes nombres qu'à l'exercice 4. Les éléments de requetes.rdd sont des objets
# Row, dont on lit les champs par leur nom (ligne.statut).

## 12. Soumettre un job

Un traitement qui tourne chaque nuit ne vit pas dans un notebook : c'est un script, soumis au cluster par `spark-submit`. Écrivez le script `rubriques.py` (la commande magique `%%writefile rubriques.py`, en première ligne de la cellule, l'enregistre au lieu de l'exécuter), qui reprend l'exercice 5 : il crée sa propre session, sans `master` (c'est `spark-submit` qui le donne), lit le journal dont le chemin est son premier argument (`sys.argv[1]`), compte les requêtes par rubrique et écrit le résultat, trié, dans le dossier de son second argument avec `saveAsTextFile`. Il arrête sa session en fin de script.

Soumettez-le ensuite, depuis une cellule, avec `!spark-submit --master local[2] rubriques.py access-small.log rubriques`, et affichez le début du résultat (`!head rubriques/part-*`). Pourquoi le résultat est-il un dossier, et pas un fichier ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le script n'a pas accès aux fonctions du notebook : il redéfinit `MOTIF`, `analyser` et `rubrique`. `saveAsTextFile` refuse d'écrire dans un dossier qui existe : `!rm -rf rubriques` avant de soumettre à nouveau. Le script tourne dans un autre processus que le notebook, avec sa propre interface web (sur le port 4041, le 4040 étant pris par la session du notebook). `spark-submit` écrit ses journaux sur la sortie d'erreur : `2> spark-submit.log` les envoie dans un fichier, et `|| tail -n 20 spark-submit.log` en affiche la fin si la soumission échoue. La commande vient avec le paquet `pyspark`, dans le dossier `bin/` de l'environnement Python : hors de Colab, activez cet environnement pour la trouver.

```python
%%writefile rubriques.py
import re
import sys

from pyspark.sql import SparkSession

MOTIF = re.compile(...)


def analyser(ligne): ...


def rubrique(chemin): ...


spark = SparkSession.builder.appName("rubriques").getOrCreate()
entree, sortie = sys.argv[1], sys.argv[2]
journal = spark.sparkContext.textFile(entree).map(analyser).filter(...)
rubriques = journal.map(...).reduceByKey(...)
rubriques.sortBy(..., ascending=False).saveAsTextFile(sortie)
spark.stop()
```

### Solution

In [ ]:
%%writefile rubriques.py
import re
import sys

from pyspark.sql import SparkSession

MOTIF = re.compile(
  r'(\S+) \S+ \S+ \[([^\]]+)\] "(\S+) (\S+) [^"]*" (\d{3}) (\d+|-) "[^"]*" "([^"]*)"'
)


def analyser(ligne):
  correspondance = MOTIF.match(ligne)
  if correspondance is None:
    return None
  ip, date, methode, chemin, statut, taille, navigateur = correspondance.groups()
  taille = 0 if taille == "-" else int(taille)
  return (ip, date, methode, chemin, int(statut), taille, navigateur)


def rubrique(chemin):
  return "/" + chemin.split("/")[1].split("?")[0]


spark = SparkSession.builder.appName("rubriques").getOrCreate()
entree, sortie = sys.argv[1], sys.argv[2]
journal = (
  spark.sparkContext.textFile(entree)
  .map(analyser)
  .filter(lambda requete: requete is not None)
)
rubriques = journal.map(lambda requete: (rubrique(requete[3]), 1)).reduceByKey(
  lambda a, b: a + b
)
rubriques.sortBy(lambda paire: paire[1], ascending=False).saveAsTextFile(sortie)
spark.stop()

In [ ]:
!rm -rf rubriques && spark-submit --master local[2] rubriques.py access-small.log rubriques 2> spark-submit.log || tail -n 20 spark-submit.log
!ls rubriques
!head -n 5 rubriques/part-00000
# Un dossier : chaque partition est écrite par sa tâche, dans son propre fichier
# (part-00000, part-00001…), sans passer par le pilote. Le fichier _SUCCESS dit que
# le job est allé au bout.

## Fin

Arrêtez la session, qui libère les exécuteurs et la mémoire du cache.

In [ ]:
spark.stop()